In [1]:
import warnings

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
)

## Enable if upsampling is used 
# from sklearn.utils import resample 
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
import pandas as pd
import numpy as np
import os
import torch
import json


## GPU/RAM Check

In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Thu Mar 19 02:38:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             46W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 189.9 gigabytes of available RAM

You are using a high-RAM runtime!


## Preparing Training Data

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
human_labeled_sentiment = pd.read_parquet('/content/drive/MyDrive/ColabThesisData/labeled.parquet')

# Replace with your dataset loading
ds = human_labeled_sentiment

In [6]:
ds.head(10)

,id,author_id,message,date_time,engagement,forum,url,company_name,ticker,message_length,year_month,year,sentiment,labeled_at
0,Sijoitustieto.comment-117218,Sijoitustieto.Unknown,"Juuri näin. Yllättävää,, että tässä tulee jatk...",2025-03-01 03:19:00,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Citycon,CTY1S,109,2025-03,2025,1,2026-03-16T00:36:15.908780
1,Sijoitustieto.comment-26243,Sijoitustieto.Unknown,Tervehdys! \n\nMukava kuulla. Tässä tullut lis...,2019-06-08 19:37:00,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Saga Furs,SAGCV,136,2019-06,2019,2,2026-03-16T00:36:23.403400
2,Kauppalehti.post-6085573,Kauppalehti.55430,Siis Hannu Lehto antoi ymmärtää. Vaikeudet on ...,2019-12-21 19:49:03,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/leht...,Lehto Group,LEHTO,197,2019-12,2019,2,2026-03-16T00:36:33.661733
3,Sijoitustieto.comment-24835,Sijoitustieto.Unknown,Ihan kiinnostavalta vaikuttaa. IPOihin en ole ...,2019-03-23 17:10:00,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Aallon Group,AALLON,216,2019-03,2019,2,2026-03-16T00:36:42.800170
4,Kauppalehti.post-6189721,Kauppalehti.55017,"Hallitushan allokaatiosta toki päättää, mutta ...",2016-09-28 18:01:56,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/vinc...,Vincit,VINCIT,265,2016-09,2016,1,2026-03-16T00:36:53.395512
5,Kauppalehti.post-6063559,Kauppalehti.981,YK jäi valitettavasti väliin mutta kiitokset s...,2017-03-27 09:57:47,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/reve...,Revenio,REG1V,2085,2017-03,2017,2,2026-03-16T09:31:22.161538
6,Sijoitustieto.comment-109908,Sijoitustieto.Unknown,Omat käynnit Tokmannilla liittyvät käytännössä...,2024-07-24 12:58:00,+4,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Tokmanni,TOKMAN,360,2024-07,2024,1,2026-03-16T09:37:35.563658
7,Kauppalehti.post-7398490,Kauppalehti.51236,Jiiärrä sanoi:\nElä hyvä mies lähde nesteeseen...,2024-05-08 07:27:55,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/kone...,Konecranes,KCR,195,2024-05,2024,1,2026-03-16T09:38:39.687686
8,Sijoitustieto.comment-26007,Sijoitustieto.Unknown,Harvia tykitti muuten aika timmin tuloksen. Tä...,2019-05-22 19:01:00,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Harvia,HARVIA,514,2019-05,2019,2,2026-03-16T09:39:00.230697
9,Kauppalehti.post-5666319,Kauppalehti.45987,> Lehdistötiedote:\n>\n> Biohit Oyj tuo markki...,2016-11-16 07:08:15,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/mita...,Biohit,BIOBV,628,2016-11,2016,1,2026-03-16T09:39:19.286718


In [7]:
ds["sentiment"].value_counts()

,count
sentiment,
1,253
2,205
0,150


## Load Pretrained Model

In [8]:
MODEL_PATH = 'FacebookAI/xlm-roberta-large'
NUM_LABELS = 3
LABEL_NAMES = ["negative", "neutral", "positive"]
MAX_SEQ_LENGTH = 512  # XLM-RoBERTa max is 512 tokens

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=NUM_LABELS,
    device_map="auto",
    dtype=torch.bfloat16,
)

model.eval()
print(f"Model loaded from: {MODEL_PATH}")
print(f"Model device: {next(model.parameters()).device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded from: FacebookAI/xlm-roberta-large
Model device: cuda:0


## Benchmark on Labeled Dataset

In [9]:
def run_benchmark(df, label_col="sentiment", batch_size=32):
    """Run model predictions on df and return metrics vs ground truth labels."""
    texts = ("yritys: " + df["company_name"] + " viesti: " + df["message"]).tolist()
    true_labels = df[label_col].tolist()
    all_preds = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        encoding = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=MAX_SEQ_LENGTH,
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].to(model.device)
        attention_mask = encoding["attention_mask"].to(model.device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1).cpu().tolist()

        all_preds.extend(preds)

    accuracy  = accuracy_score(true_labels, all_preds)
    precision = precision_score(true_labels, all_preds, average="weighted", zero_division=0)
    recall    = recall_score(true_labels, all_preds, average="weighted", zero_division=0)
    f1        = f1_score(true_labels, all_preds, average="weighted", zero_division=0)

    metrics = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "n_samples": len(texts),
    }
    return metrics, all_preds


metrics, predictions = run_benchmark(ds)

print("=" * 50)
print("BENCHMARK RESULTS ON LABELED DATASET")
print("=" * 50)
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

BENCHMARK RESULTS ON LABELED DATASET
  accuracy: 0.3372
  precision: 0.1137
  recall: 0.3372
  f1: 0.1700
  n_samples: 608


In [10]:
from sklearn.metrics import classification_report, confusion_matrix

print("Per-class report:")
print(classification_report(ds["sentiment"], predictions, target_names=LABEL_NAMES))

print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(
    confusion_matrix(ds["sentiment"], predictions),
    index=[f"true_{l}" for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))

Per-class report:
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00       150
     neutral       0.00      0.00      0.00       253
    positive       0.34      1.00      0.50       205

    accuracy                           0.34       608
   macro avg       0.11      0.33      0.17       608
weighted avg       0.11      0.34      0.17       608

Confusion matrix (rows=true, cols=predicted):
               pred_negative  pred_neutral  pred_positive
true_negative              0             0            150
true_neutral               0             0            253
true_positive              0             0            205


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Fine-tune on Financial Domain Data

In [11]:
# 80/20 stratified split — holdout never seen during fine-tuning
ft_train_df, ft_holdout_df = train_test_split(
    ds[["message", "sentiment", "company_name"]],
    test_size=0.2,
    random_state=42,
    stratify=ds["sentiment"],
)

print(f"Fine-tune train: {len(ft_train_df)}  |  Holdout: {len(ft_holdout_df)}")
print("Train distribution:\n", ft_train_df["sentiment"].value_counts().sort_index())
print("Holdout distribution:\n", ft_holdout_df["sentiment"].value_counts().sort_index())

Fine-tune train: 486  |  Holdout: 122
Train distribution:
 sentiment
0    120
1    202
2    164
Name: count, dtype: int64
Holdout distribution:
 sentiment
0    30
1    51
2    41
Name: count, dtype: int64


In [12]:
def make_hf_dataset(df):
    df = df.copy()
    df["text"] = "yritys: " + df["company_name"] + " viesti: " + df["message"]
    hf = Dataset.from_pandas(df[["text", "sentiment"]].rename(columns={"sentiment": "label"}).reset_index(drop=True))
    return hf.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH),
        batched=True,
    )

ft_train_ds   = make_hf_dataset(ft_train_df)
ft_holdout_ds = make_hf_dataset(ft_holdout_df)

Map:   0%|          | 0/486 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

In [13]:
FINETUNED_MODEL_PATH = '/content/drive/MyDrive/ColabThesisData/final_model_finetuned/'

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
    }

ft_training_args = TrainingArguments(
    output_dir='/tmp/ft_checkpoints/',   # ephemeral — keeps Drive clean
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

ft_trainer = Trainer(
    model=model,
    args=ft_training_args,
    train_dataset=ft_train_ds,
    eval_dataset=ft_holdout_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

ft_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,1.122182,0.336066,0.169064,0.112940,0.336066
2,No log,1.091444,0.393443,0.339692,0.311310,0.393443
3,No log,1.079790,0.385246,0.307752,0.272131,0.385246
4,No log,1.079566,0.368852,0.296734,0.260746,0.368852
5,No log,1.078381,0.368852,0.296734,0.260746,0.368852


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=155, training_loss=1.1061819753339213, metrics={'train_runtime': 59.284, 'train_samples_per_second': 40.989, 'train_steps_per_second': 2.615, 'total_flos': 1739546776116612.0, 'train_loss': 1.1061819753339213, 'epoch': 5.0})

In [14]:
# Save fine-tuned model
ft_trainer.save_model(FINETUNED_MODEL_PATH)
tokenizer.save_pretrained(FINETUNED_MODEL_PATH)
print(f"Fine-tuned model saved to: {FINETUNED_MODEL_PATH}")

# Evaluate on holdout
holdout_results = ft_trainer.predict(ft_holdout_ds)
ft_preds = np.argmax(holdout_results.predictions, axis=1)
ft_true = ft_holdout_df["sentiment"].tolist()

print("\n" + "=" * 50)
print("HOLDOUT RESULTS AFTER FINE-TUNING")
print("=" * 50)
print(classification_report(ft_true, ft_preds, target_names=LABEL_NAMES))

print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(
    confusion_matrix(ft_true, ft_preds),
    index=[f"true_{l}" for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved to: /content/drive/MyDrive/ColabThesisData/final_model_finetuned/



HOLDOUT RESULTS AFTER FINE-TUNING
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00        30
     neutral       0.45      0.45      0.45        51
    positive       0.32      0.56      0.41        41

    accuracy                           0.38       122
   macro avg       0.26      0.34      0.29       122
weighted avg       0.30      0.38      0.33       122

Confusion matrix (rows=true, cols=predicted):
               pred_negative  pred_neutral  pred_positive
true_negative              0            10             20
true_neutral               0            23             28
true_positive              0            18             23


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Augmented Training: Human + LLM Labels

In [15]:
LLM_LABELS_PATH = '/content/drive/MyDrive/ColabThesisData/llm_labeled.parquet'

llm_labels = pd.read_parquet(LLM_LABELS_PATH)

# Safety: recover holdout IDs from the original ds index and exclude them
holdout_ids = set(ds.loc[ft_holdout_df.index, 'id'])
leaked = llm_labels[llm_labels['id'].isin(holdout_ids)]
if len(leaked):
    print(f"WARNING: {len(leaked)} LLM labels overlap with holdout — dropping them.")
    llm_labels = llm_labels[~llm_labels['id'].isin(holdout_ids)]

# Combine: human train split + all LLM labels (keep company_name for text formatting)
ft_train_augmented_df = pd.concat(
    [ft_train_df, llm_labels[["message", "sentiment", "company_name"]]],
    ignore_index=True,
)

print(f"Train  : {len(ft_train_df)} human  →  {len(ft_train_augmented_df)} total (+{len(llm_labels)} LLM)")
print(f"Holdout: {len(ft_holdout_df)} (unchanged)")
print("\nAugmented train sentiment distribution:")
print(ft_train_augmented_df["sentiment"].value_counts().sort_index().rename({0: "negative", 1: "neutral", 2: "positive"}))

Train  : 486 human  →  10486 total (+10000 LLM)
Holdout: 122 (unchanged)

Augmented train sentiment distribution:
sentiment
negative    3711
neutral     2984
positive    3791
Name: count, dtype: int64


In [16]:
# Reload base model fresh — same starting point as before, not the already fine-tuned weights
model_aug = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=NUM_LABELS,
    device_map="auto",
    dtype=torch.bfloat16,
)

ft_train_augmented_ds = make_hf_dataset(ft_train_augmented_df)
# ft_holdout_ds is unchanged

AUGMENTED_MODEL_PATH = '/content/drive/MyDrive/ColabThesisData/final_model_finetuned_augmented/'

aug_training_args = TrainingArguments(
    output_dir='/tmp/aug_checkpoints/',
    num_train_epochs=15,         # more data needs more epochs to converge
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,          # slightly higher — larger dataset can support it
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,            # log training loss every 50 steps so we can see it move
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

aug_trainer = Trainer(
    model=model_aug,
    args=aug_training_args,
    train_dataset=ft_train_augmented_ds,
    eval_dataset=ft_holdout_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

aug_trainer.train()

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/10486 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.884568,1.008745,0.565574,0.562764,0.636635,0.565574
2,0.793474,0.889642,0.639344,0.633678,0.666350,0.639344
3,0.756628,1.030462,0.590164,0.565428,0.695708,0.590164
4,0.731164,0.955690,0.606557,0.597843,0.652038,0.606557
5,0.653717,0.970254,0.573770,0.563837,0.608237,0.573770
6,0.628983,0.970098,0.614754,0.607145,0.650944,0.614754
7,0.608317,0.938837,0.622951,0.614451,0.657533,0.622951
8,0.633891,1.003710,0.614754,0.607203,0.651601,0.614754
9,0.639190,1.071162,0.557377,0.529971,0.616258,0.557377
10,0.595621,1.026037,0.598361,0.585797,0.640060,0.598361


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=9840, training_loss=0.6633557598765304, metrics={'train_runtime': 989.5799, 'train_samples_per_second': 158.946, 'train_steps_per_second': 9.944, 'total_flos': 1.0814022828002709e+17, 'train_loss': 0.6633557598765304, 'epoch': 15.0})

In [17]:
aug_trainer.save_model(AUGMENTED_MODEL_PATH)
tokenizer.save_pretrained(AUGMENTED_MODEL_PATH)
print(f"Augmented model saved to: {AUGMENTED_MODEL_PATH}")

aug_results = aug_trainer.predict(ft_holdout_ds)
aug_preds = np.argmax(aug_results.predictions, axis=1)
ft_true   = ft_holdout_df["sentiment"].tolist()

print("\n" + "=" * 50)
print("HOLDOUT RESULTS — AUGMENTED (human + LLM)")
print("=" * 50)
print(classification_report(ft_true, aug_preds, target_names=LABEL_NAMES))

print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(
    confusion_matrix(ft_true, aug_preds),
    index=[f"true_{l}" for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))

# Side-by-side summary vs the human-only run
print("\n── Accuracy comparison ──")
print(f"  Human only  (~486 train) : {accuracy_score(ft_true, ft_preds):.3f}")
print(f"  Human + LLM (~{len(ft_train_augmented_df)} train) : {accuracy_score(ft_true, aug_preds):.3f}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Augmented model saved to: /content/drive/MyDrive/ColabThesisData/final_model_finetuned_augmented/



HOLDOUT RESULTS — AUGMENTED (human + LLM)
              precision    recall  f1-score   support

    negative       0.53      0.57      0.55        30
     neutral       0.79      0.51      0.62        51
    positive       0.61      0.85      0.71        41

    accuracy                           0.64       122
   macro avg       0.64      0.64      0.63       122
weighted avg       0.67      0.64      0.63       122

Confusion matrix (rows=true, cols=predicted):
               pred_negative  pred_neutral  pred_positive
true_negative             17             6              7
true_neutral              10            26             15
true_positive              5             1             35

── Accuracy comparison ──
  Human only  (~486 train) : 0.377
  Human + LLM (~10486 train) : 0.639
